# One ranged reader over the shelf, for N sessions at once

Shane's ruling of 2026-11-17: the `/files/` file system must not put all files in memory, so that several concurrent
sessions stay usable. This lab measures the reader the amendment specifies against the eager mount it replaces.

`measure-ranged.mjs` loads a probe extension into the e2e harness's Chromium. An extension page stands in for the
offscreen document (which owns the shelf), and one dedicated worker per session stands in for the Pyodide worker. Each
session's directory is two files of 128 MiB of random bytes, kept as IndexedDB Blobs, as the value store keeps an
attachment. Two ways to read it, for 1, 4 and 8 sessions at once:

- **lazy**: every file read through in 512 KiB `Blob.slice(a, b).arrayBuffer()` chunks, each chunk transferred to the
  session's worker, which keeps at most 32 chunks (16 MiB), the mount's bounded cache;
- **eager**: every file loaded whole as an ArrayBuffer and held, as the ruled MEMFS mount holds the directory.

Each configuration runs in a FRESH browser over one shared profile, so its baseline is not what an earlier one left
behind (a first version ran them all in one browser, and the baseline climbed from 1.0 to 3.0 GiB; its figures are
void). Every configuration runs three times, interleaved. Memory is the harness browser's process tree, sampled from
outside every 100 ms, with the macOS memory pressure level and the kernel's free-memory percentage. Under pressure
Chrome purges and the OS compresses, so RSS reads LOW, in the flattering direction: a run that leaves the normal
level is VOID and left out of every figure. A lazy run reads its directory five times over, so it lasts long enough
for the sampler to see its steady state.

The machine was shared: unit suites from other sessions may run (no other Playwright Chromium did). **Every latency
and throughput figure here is taken under that load**, and no spec sentence rests on one. It is a lab, not the product: the
channel to Python (route A's sliced wait) is not in it, since `notebooks/pyodide-mount/` measured that (0.26 ms a
request end to end).

In [1]:
import json
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "session-store"
DATA = HERE / "data"
D = json.loads((DATA / "ranged.json").read_text())
G = json.loads((DATA / "ranged-gc.json").read_text())
pd.set_option("display.width", 200)
print(pd.DataFrame([{"file": f"data/{f}", "sha256": wmlnb.sha256_file(DATA / f)[:16]} for f in ["ranged.json", "ranged-gc.json"]]).to_string(index=False))
m = D["machine"]
print(f"browser: Chromium {D['browser']}; {m['cpus']}, {m['ramGiB']} GiB RAM; at {D['at']}; quick={D['quick']}")
print(f"load average before {m['load']}, after {m['loadAfter']}")
print("other Playwright browsers on the machine at start / end:", D.get("otherBrowsersAtStart"), "/", D.get("otherBrowsersAtEnd"))
print(f"each session: {D['files']} files x {D['fileMiB']} MiB; chunk {D['chunkBytes'] // 1024} KiB; cache {D['cacheChunks']} chunks; {D['reps']} runs per configuration; lazy reads {D['lazyPasses']} passes")
print("other test-slot jobs at start:", D.get("otherTestSlotJobsAtStart"), "; timings under load:", D.get("timingsUnderLoad"))

               file           sha256
   data/ranged.json ac49b38acfd26826
data/ranged-gc.json cedeb48ccfaff962
browser: Chromium 151.0.7922.34; Apple M5 Pro, 24 GiB RAM; at 2026-11-18T00:19:21.983Z; quick=False
load average before [2.63720703125, 2.09619140625, 2.39501953125], after [2.68603515625, 2.2958984375, 2.42578125]
other Playwright browsers on the machine at start / end: 0 / 0
each session: 2 files x 128 MiB; chunk 512 KiB; cache 32 chunks; 3 runs per configuration; lazy reads 5 passes
other test-slot jobs at start: 0 ; timings under load: False


## 1. Memory, lazy against eager

`grew MiB` is the peak of the browser tree's RSS during a run, less its value just before. Per configuration: the
median of the valid runs, and their range. The last columns are throughput, taken under load.

In [2]:
runs = pd.DataFrame([{**r, "grew": r["rssPeakMiB"] - r["rssBaseMiB"]} for r in D["runs"]])
print("runs:", len(runs), "; void (memory pressure left the normal level):", int((~runs["valid"]).sum()))
void = runs[~runs["valid"]][["mode", "sessions", "rep", "maxPressureLevel", "minFreePct"]]
if len(void): print(void.to_string(index=False))
ok = runs[runs["valid"]]
agg = ok.groupby(["mode", "sessions"]).agg(valid_runs=("grew", "size"), dir_MiB_each=("dirMiBPerSession", "first"),
    grew_median=("grew", "median"), grew_min=("grew", "min"), grew_max=("grew", "max"),
    free_pct_min=("minFreePct", "min"), MBps_median=("mbPerS", "median"), chunk_ms_median=("chunkMsMedian", "median")).reset_index()
print(agg.to_string(index=False))

runs: 18 ; void (memory pressure left the normal level): 5
 mode  sessions  rep  maxPressureLevel  minFreePct
 lazy         4    0                 2          34
eager         4    0                 2          45
 lazy         8    0                 2          33
 lazy         8    1                 2          34
 lazy         8    2                 2          34
 mode  sessions  valid_runs  dir_MiB_each  grew_median  grew_min  grew_max  free_pct_min  MBps_median  chunk_ms_median
eager         1           3           256        262.0       261       262            47          NaN              NaN
eager         4           2           256       1032.5      1032      1033            65          NaN              NaN
eager         8           3           256       2058.0      2058      2059            62          NaN              NaN
 lazy         1           3           256       1078.0      1078      1079            47       2147.0              0.2
 lazy         4           2           25

## 1b. What the lazy reader HOLDS: the same lazy runs, collected as they go

Section 1's lazy peak grows with the bytes READ, not with the cache: one session reading 1.25 GiB through a 16 MiB
cache grew by about the bytes it read. Each chunk is a fresh ArrayBuffer, and V8 had not collected the dropped ones
before the run ended, so that figure is garbage not yet collected, not what the reader holds. A real reader copies into a
fixed buffer (route A's shared one), which makes no such garbage. To measure what the reader holds, `--gc` runs the
same lazy configurations in a browser started with `--expose-gc`, the page calling `gc()` after each file and each
worker after every 32 chunks. Its eager runs are section 1's: an eager load holds what it loaded, collected or not.

In [3]:
gr = pd.DataFrame([{**r, "grew": r["rssPeakMiB"] - r["rssBaseMiB"]} for r in G["runs"]])
print("gc runs:", len(gr), "; void:", int((~gr["valid"]).sum()), "; timings under load:", G.get("timingsUnderLoad"), "; other test-slot jobs at start:", G.get("otherTestSlotJobsAtStart"))
gok = gr[gr["valid"]]
gagg = gok.groupby("sessions").agg(valid_runs=("grew", "size"), grew_median=("grew", "median"), grew_min=("grew", "min"), grew_max=("grew", "max")).reset_index()
eager = agg[agg["mode"] == "eager"][["sessions", "grew_median"]].rename(columns={"grew_median": "eager_grew_median"})
both = gagg.merge(eager, on="sessions", how="left")
both["lazy_per_session"] = (both["grew_median"] / both["sessions"]).round(0)
both["eager_per_session"] = (both["eager_grew_median"] / both["sessions"]).round(0)
print(both.to_string(index=False))

gc runs: 9 ; void: 0 ; timings under load: True ; other test-slot jobs at start: 3
 sessions  valid_runs  grew_median  grew_min  grew_max  eager_grew_median  lazy_per_session  eager_per_session
        1           3         51.0        50        55              262.0              51.0              262.0
        4           3        169.0       165       170             1032.5              42.0              258.0
        8           3        219.0       185       312             2058.0              27.0              257.0


## 2. Small files: what eager saves

200 files of 4 KiB, each read whole once: lazily (one slice each), against loading every file first and reading from
memory. Route A's channel adds about 0.26 ms a request on top of the lazy figure for a Python cell.

In [4]:
s = D["small"]
lazy_per = s["lazyReadAllMs"] / s["files"]
eager_per = (s["eagerLoadMs"] + s["eagerReadMs"]) / s["files"]
print(pd.DataFrame([{"files": s["files"], "bytes each": s["fileBytes"], "lazy ms (all)": s["lazyReadAllMs"], "eager load ms (all)": s["eagerLoadMs"],
                     "eager read ms (all)": s["eagerReadMs"], "lazy ms per file": round(lazy_per, 3), "eager ms per file": round(eager_per, 3),
                     "lazy + route A channel ms per file": round(lazy_per + 0.26, 3)}]).to_string(index=False))

 files  bytes each  lazy ms (all)  eager load ms (all)  eager read ms (all)  lazy ms per file  eager ms per file  lazy + route A channel ms per file
   200        4096           24.9                 12.3                  0.1             0.124              0.062                               0.385


## What the amendment takes from this

- **Lazy only, no eager mount under any size.** Collected as it goes, the lazy reader held about 50 MiB for one
  session and about 220 MiB for eight (section 1b): about its 16 MiB cache per session plus a fixed cost, and
  sublinear in sessions. The eager load grew by its directory's size for every session (about 262 MiB each, 2 GiB for
  eight), which is the cost the ruling forbids: it scales with what the sessions hold, not with what they read.
- **What eager would save is small.** For small files (section 2) it saves the per-file slice, a fraction of a
  millisecond, plus route A's 0.26 ms a request: an "eager under T bytes" rule would buy that and cost a second code
  path and a mount whose memory depends on what a session holds. Not worth it.
- **What the spec may say about memory:** the reader's resident cost is its cache (16 MiB a session here) plus a fixed
  overhead, independent of the directory's size; a reader that allocates a buffer per chunk leaves garbage up to the
  bytes it read until V8 collects it, so the reader copies into a fixed buffer. No latency or throughput figure here
  is used: both runs were taken while other test runs shared the machine.
- The plain lazy runs at eight sessions were all void (the memory pressure left the normal level), which is itself the
  uncollected garbage pushing the machine: a reason for the fixed buffer, not a figure.